In [1]:
!pip install Groq

In [2]:
!pip install langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 14.0 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-groq]


In [5]:
import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Get API Key
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# Send messages
response = llm.invoke([
    SystemMessage(content="Act as a professional"),
    HumanMessage(content="tell me my name Gokul")
])

# Display output
print(response.content)

Enter your Groq API Key: ··········
Your name is Gokul.


In [7]:
import os
import json
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# --------------------------------
# 1. API KEY
# --------------------------------

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# --------------------------------
# 2. INITIALIZE LLM
# --------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# --------------------------------
# 3. LONG-TERM MEMORY
# --------------------------------

memory_file = "user_memory.json"

if os.path.exists(memory_file):
    with open(memory_file, "r") as file:
        user_memory = json.load(file)
else:
    user_memory = {}

# --------------------------------
# 4. CONVERSATION HISTORY
# --------------------------------

chat_history = []

# --------------------------------
# 5. CHAT FUNCTION
# --------------------------------

def chat(user_input):

    global user_memory

    system_prompt = f"""
    You are a professional AI assistant.

    User information stored in memory:
    {json.dumps(user_memory)}

    Instructions:
    1. Remember information shared by the user.
    2. If the user tells you their name, identify it.
    3. Use the user's name when relevant.
    4. Answer naturally and professionally.
    5. Never invent personal information.
    """

    messages = [
        SystemMessage(content=system_prompt)
    ]

    messages.extend(chat_history)

    messages.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(messages)

    # --------------------------------
    # 6. EXTRACT USER INFORMATION
    # --------------------------------

    extraction_prompt = f"""
    Analyze the following user message.

    Current memory:
    {json.dumps(user_memory)}

    User message:
    {user_input}

    Extract any personal information explicitly shared by the user.

    Return ONLY valid JSON.

    Example:
    {{"name": "Dharani"}}

    If no new personal information is shared, return:
    {{}}

    Do not guess or invent information.
    """

    extraction_response = llm.invoke(extraction_prompt)

    try:
        new_memory = json.loads(extraction_response.content)

        # Update memory
        user_memory.update(new_memory)

        # Save permanently
        with open(memory_file, "w") as file:
            json.dump(user_memory, file, indent=4)

    except json.JSONDecodeError:
        print("Memory extraction failed.")

    # --------------------------------
    # 7. SAVE CHAT HISTORY
    # --------------------------------

    chat_history.append(
        HumanMessage(content=user_input)
    )

    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content


# --------------------------------
# 8. INTERACTIVE CHATBOT
# --------------------------------

print("AI Assistant Started!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye!")
        break

    answer = chat(user_input)

    print("\nAI:", answer)
    print()

Enter your Groq API Key: ··········
AI Assistant Started!
Type 'exit' to stop.

You: what is my name

AI: Your name is **Jim Rat**.

You: exit
AI: Goodbye!


In [9]:
!pip install -q "mcp>=2"

In [13]:
try:
    from mcp.server.mcpserver import MCPServer      # path from the error message
except ImportError:
    from mcp.server import MCPServer                # path from the tutorial
from mcp.server.mcpserver.exceptions import ToolError

mcp = MCPServer("nutrition")

In [14]:
print(await lookup_food("idli", 200))
print(await log_meal("breakfast", "idli", 200))
print(await daily_summary())

idli (200 g): 260.0 kcal, protein 8.0 g, carbs 52.0 g, fat 0.8 g, fiber 2.0 g
Logged 200 g idli for breakfast (260.0 kcal).
Today:
- breakfast: 200 g idli (260.0 kcal)
Total: 260.0 kcal, protein 8.0 g, carbs 52.0 g, fat 0.8 g


In [17]:
"""Nutrition MCP server (stdio). Mirrors the structure of the MCP weather tutorial.

Setup:
    uv init nutrition && cd nutrition
    uv venv && source .venv/bin/activate
    uv add "mcp[cli]"
    # put this file in the folder as nutrition.py, then:
    uv run nutrition.py
"""

import json
import logging
from datetime import date
from pathlib import Path

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

logger = logging.getLogger(__name__)      # stderr only: never print() in a stdio server

mcp = MCPServer("nutrition")

DIARY_FILE = Path.home() / ".nutrition_mcp" / "diary.json"

# Per 100 g (approximate). Extend freely.
FOODS = {
    "rice (cooked)":   {"kcal": 130, "protein": 2.7,  "carbs": 28.0, "fat": 0.3,  "fiber": 0.4},
    "chapati":         {"kcal": 297, "protein": 9.0,  "carbs": 46.0, "fat": 7.5,  "fiber": 4.0},
    "idli":            {"kcal": 130, "protein": 4.0,  "carbs": 26.0, "fat": 0.4,  "fiber": 1.0},
    "dosa":            {"kcal": 168, "protein": 3.9,  "carbs": 29.0, "fat": 3.7,  "fiber": 1.2},
    "sambar":          {"kcal": 50,  "protein": 2.5,  "carbs": 8.0,  "fat": 1.0,  "fiber": 2.0},
    "dal (cooked)":    {"kcal": 116, "protein": 9.0,  "carbs": 20.0, "fat": 0.4,  "fiber": 8.0},
    "chicken breast":  {"kcal": 165, "protein": 31.0, "carbs": 0.0,  "fat": 3.6,  "fiber": 0.0},
    "egg (boiled)":    {"kcal": 155, "protein": 13.0, "carbs": 1.1,  "fat": 11.0, "fiber": 0.0},
    "fish (cooked)":   {"kcal": 206, "protein": 22.0, "carbs": 0.0,  "fat": 12.0, "fiber": 0.0},
    "paneer":          {"kcal": 265, "protein": 18.0, "carbs": 1.2,  "fat": 20.0, "fiber": 0.0},
    "curd":            {"kcal": 61,  "protein": 3.5,  "carbs": 4.7,  "fat": 3.3,  "fiber": 0.0},
    "milk":            {"kcal": 61,  "protein": 3.2,  "carbs": 4.8,  "fat": 3.3,  "fiber": 0.0},
    "banana":          {"kcal": 89,  "protein": 1.1,  "carbs": 23.0, "fat": 0.3,  "fiber": 2.6},
    "apple":           {"kcal": 52,  "protein": 0.3,  "carbs": 14.0, "fat": 0.2,  "fiber": 2.4},
    "potato (boiled)": {"kcal": 87,  "protein": 1.9,  "carbs": 20.0, "fat": 0.1,  "fiber": 1.8},
    "peanuts":         {"kcal": 567, "protein": 26.0, "carbs": 16.0, "fat": 49.0, "fiber": 8.5},
    "oats":            {"kcal": 389, "protein": 17.0, "carbs": 66.0, "fat": 7.0,  "fiber": 10.6},
    "tofu":            {"kcal": 76,  "protein": 8.0,  "carbs": 1.9,  "fat": 4.8,  "fiber": 0.3},
}


# ----------------------------- helpers -----------------------------------
def load_diary() -> dict:
    try:
        return json.loads(DIARY_FILE.read_text())
    except Exception:
        return {}


def save_diary(diary: dict) -> None:
    DIARY_FILE.parent.mkdir(parents=True, exist_ok=True)
    DIARY_FILE.write_text(json.dumps(diary, indent=2))


def find_food(name: str) -> str | None:
    key = name.lower().strip()
    return next((f for f in FOODS if key == f or key in f or f in key), None)


def scaled(food: str, grams: float) -> dict:
    return {k: round(v * grams / 100, 1) for k, v in FOODS[food].items()}


# ------------------------------- tools -----------------------------------
@mcp.tool()
async def lookup_food(name: str, grams: float = 100) -> str:
    """Get nutrition facts for a food.

    Args:
        name: Food name, e.g. "idli" or "chicken breast"
        grams: Portion size in grams (default 100)
    """
    match = find_food(name)
    if not match:
        raise ToolError(f"'{name}' is not in the database. Known foods: {', '.join(FOODS)}")
    n = scaled(match, grams)
    return (f"{match} ({grams:g} g): {n['kcal']} kcal, protein {n['protein']} g, "
            f"carbs {n['carbs']} g, fat {n['fat']} g, fiber {n['fiber']} g")


@mcp.tool()
async def log_meal(meal: str, food: str, grams: float) -> str:
    """Log a food to today's diary.

    Args:
        meal: breakfast, lunch, dinner or snack
        food: Food name from the database
        grams: Portion size in grams
    """
    match = find_food(food)
    if not match:
        raise ToolError(f"'{food}' is not in the database.")
    n = scaled(match, grams)
    diary = load_diary()
    diary.setdefault(str(date.today()), []).append(
        {"meal": meal, "food": match, "grams": grams,
         "kcal": n["kcal"], "protein": n["protein"], "carbs": n["carbs"], "fat": n["fat"]})
    save_diary(diary)
    return f"Logged {grams:g} g {match} for {meal} ({n['kcal']} kcal)."


@mcp.tool()
async def daily_summary() -> str:
    """Show today's totals and how they compare with saved targets."""
    diary = load_diary()
    entries = diary.get(str(date.today()), [])
    if not entries:
        return "Nothing logged today."
    t = {k: round(sum(e[k] for e in entries), 1) for k in ("kcal", "protein", "carbs", "fat")}
    lines = [f"- {e['meal']}: {e['grams']:g} g {e['food']} ({e['kcal']} kcal)" for e in entries]
    out = ["Today:", *lines,
           f"Total: {t['kcal']} kcal, protein {t['protein']} g, carbs {t['carbs']} g, fat {t['fat']} g"]
    tg = diary.get("_targets")
    if tg:
        out.append(f"Target: {tg['kcal']} kcal -> {round(tg['kcal'] - t['kcal'])} kcal remaining")
    return "\n".join(out)


@mcp.tool()
async def calculate_targets(age: int, sex: str, height_cm: float, weight_kg: float,
                            activity: str = "moderate", goal: str = "maintain") -> str:
    """Calculate and save daily calorie and macro targets.

    Args:
        age: Age in years
        sex: male or female
        height_cm: Height in centimetres
        weight_kg: Weight in kilograms
        activity: sedentary, light, moderate, active or very_active
        goal: lose, maintain or gain
    """
    factors = {"sedentary": 1.2, "light": 1.375, "moderate": 1.55, "active": 1.725, "very_active": 1.9}
    if activity not in factors or goal not in ("lose", "maintain", "gain"):
        raise ToolError("Invalid activity or goal value.")
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + (5 if sex.lower().startswith("m") else -161)
    tdee = bmr * factors[activity]
    kcal = max(tdee + {"lose": -500, "maintain": 0, "gain": 400}[goal], 1200)
    protein = weight_kg * (1.4 if goal == "maintain" else 1.8)
    fat = kcal * 0.25 / 9
    carbs = (kcal - protein * 4 - fat * 9) / 4
    targets = {"kcal": round(kcal), "protein": round(protein), "carbs": round(carbs), "fat": round(fat)}
    diary = load_diary()
    diary["_targets"] = targets
    save_diary(diary)
    return (f"BMR {round(bmr)} kcal, TDEE {round(tdee)} kcal. Daily target: {targets['kcal']} kcal, "
            f"protein {targets['protein']} g, carbs {targets['carbs']} g, fat {targets['fat']} g.")


# if __name__ == "__main__":
#     mcp.run(transport="stdio")
print(await lookup_food("idli", 200))
print(await log_meal("breakfast", "idli", 200))
print(await daily_summary())
print(await calculate_targets(28, "male", 172, 70, "moderate", "lose"))

idli (200 g): 260.0 kcal, protein 8.0 g, carbs 52.0 g, fat 0.8 g, fiber 2.0 g
Logged 200 g idli for breakfast (260.0 kcal).
Today:
- breakfast: 200 g idli (260.0 kcal)
- breakfast: 200 g idli (260.0 kcal)
Total: 520.0 kcal, protein 16.0 g, carbs 104.0 g, fat 1.6 g
BMR 1640 kcal, TDEE 2542 kcal. Daily target: 2042 kcal, protein 126 g, carbs 257 g, fat 57 g.


In [30]:
import traceback

def show(e):
    if isinstance(e, BaseExceptionGroup):
        for sub in e.exceptions:
            show(sub)
    else:
        traceback.print_exception(e)

try:
    await analyze_with_mcp(food_name.strip())
except BaseException as e:
    show(e)

Traceback (most recent call last):
  File "/tmp/ipykernel_5185/1072082340.py", line 153, in analyze_with_mcp
    if result.isError:
       ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pydantic/main.py", line 1110, in __getattr__
    raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')
AttributeError: 'CallToolResult' object has no attribute 'isError'. Did you mean: 'is_error'?


In [34]:
import os, json
from getpass import getpass
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY") or getpass("Groq API key: "))
MODEL = "openai/gpt-oss-120b"

FOODS = {  # per 100 g
    "egg": {"kcal": 155, "protein": 13.0, "carbs": 1.1, "fat": 11.0},
    "idli": {"kcal": 130, "protein": 4.0, "carbs": 26.0, "fat": 0.4},
    "banana": {"kcal": 89, "protein": 1.1, "carbs": 23.0, "fat": 0.3},
    "rice": {"kcal": 130, "protein": 2.7, "carbs": 28.0, "fat": 0.3},
}

def lookup_food(name: str, grams: float = 100):
    key = name.lower().strip()
    hit = next((f for f in FOODS if f in key or key in f), None)
    if not hit:
        return {"found": False, "note": "Not in database; estimate and say so."}
    return {"found": True, "food": hit, "grams": grams,
            **{k: round(v * grams / 100, 1) for k, v in FOODS[hit].items()}}

TOOLS = [{"type": "function", "function": {
    "name": "lookup_food",
    "description": "Nutrition facts for a food and portion in grams.",
    "parameters": {"type": "object", "properties": {
        "name": {"type": "string"}, "grams": {"type": "number"}}, "required": ["name"]}}}]

def analyze(food):
    msgs = [{"role": "system", "content": "You are a nutritionist. Use lookup_food, then give a short verdict."},
            {"role": "user", "content": f"Analyze: {food}"}]
    for _ in range(4):
        m = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS).choices[0].message
        if not m.tool_calls:
            return m.content
        msgs.append({"role": "assistant", "content": m.content or "",
                     "tool_calls": [{"id": t.id, "type": "function",
                                     "function": {"name": t.function.name, "arguments": t.function.arguments}}
                                    for t in m.tool_calls]})
        for t in m.tool_calls:
            res = lookup_food(**json.loads(t.function.arguments or "{}"))
            msgs.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(res)})

print(analyze(input("Enter a food name: ")))

Enter a food name: chiken
**Estimated nutrition for 100 g raw, skin‑less chicken breast** (not in database, so this is an estimate):

| Nutrient | Amount |
|----------|--------|
| **Calories** | ~165 kcal |
| **Protein** | ~31 g |
| **Total Fat** | ~3.6 g (≈1 g saturated) |
| **Carbohydrates** | 0 g |
| **Cholesterol** | ~85 mg |
| **Sodium** | ~70 mg |
| **Potassium** | ~256 mg |
| **Iron** | ~0.9 mg |
| **Vitamin B‑6** | ~0.5 mg |
| **Niacin (B3)** | ~13 mg |

**Verdict**  
Chicken breast is a lean, high‑protein food with very low carbs and modest fat. It’s a solid choice for muscle‑building, weight‑management, and overall balanced diets, especially when cooked without added fat or heavy sauces.


In [33]:
import traceback
def show(e):
    for s in (e.exceptions if isinstance(e, BaseExceptionGroup) else [e]):
        show(s) if isinstance(s, BaseExceptionGroup) else traceback.print_exception(s)

try:
    await analyze_with_mcp("egg")
except BaseException as e:
    show(e)

Traceback (most recent call last):
  File "/tmp/ipykernel_5185/1072082340.py", line 153, in analyze_with_mcp
    if result.isError:
       ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pydantic/main.py", line 1110, in __getattr__
    raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')
AttributeError: 'CallToolResult' object has no attribute 'isError'. Did you mean: 'is_error'?
